# Postprocessing All grids

In [ ]:
#0. Generate the outout variables per year for each grid with the reconstructed spectrum
# example: /home/grupos/geocean/montanoj/ShoreShop2026/grid1/outputs/output_variables


### 1. Check the individual variables do not contain NaNs or zeros.

In [ ]:
from utils.postprocessing_all_grids import run_hs_postprocessing

summary_df = run_hs_postprocessing(
    data_dir="/home/grupos/geocean/montanoj/ShoreShop2026/grid3/outputs/output_variables/hs",  # or any other grid path
    var_name="hs",
    time_name="time",
    rolling_window=30,
    rolling_std_eps=1e-6,
    monthly_var_eps=1e-4,
    save_csv_path=None,   # or a path to save CSV
    verbose=True,         # False to skip printing/display
)

### 2. Convert float 32

In [ ]:
from utils.postprocessing_all_grids import convert_output_variables_to_float32

convert_output_variables_to_float32(
    base_dir="/home/grupos/geocean/montanoj/ShoreShop2026/grid1/outputs/output_variables",
    out_root=None,
    overwrite=False,
    compress=True,
    complevel=4,
    variable=None,
    year=None,      # or a specific year like 1980
)


### 3. Concatenate all years by variable by grid

In [ ]:
from pathlib import Path
from utils.postprocessing_all_grids import concatenate_float32_grids_by_variable

# Define paths
BASE_PATH = Path("/home/grupos/geocean/montanoj/ShoreShop2026")
MERGED_OUTPUT_DIR = BASE_PATH / "outputs"  # or your output directory

# Define grids (example - adjust based on your actual grid configuration)
GRIDS = {
    "grid1": "grid1",
    # "grid2": "grid2",
    # "grid3": "grid3",
    # "grid4": "grid4",
}
# Run concatenation for float32 files
concatenate_float32_grids_by_variable(
    base_path=BASE_PATH,
    grids=GRIDS,
    output_dir=MERGED_OUTPUT_DIR,
    reference_grid="grid1",  # Grid to use for discovering variable names
    variable=['spr0','spr1','spr2','spr3'],
    verbose=True,
   
)

### 4. Filter data to avoid points in land, lagoons, and reshape to an uniform grid

In [ ]:
from utils.postprocessing_all_grids import crop_concatenated_files_by_spatial_mask

# Define your buoys
buoys = {
    # Southern buoys
    'SSBN7': (-78.484, 33.838),
    '41119': (-78.483, 33.842),
    'OCPN7': (-78.147, 33.911),
    '41108': (-78.016, 33.721), 
    '41013': (-77.764, 33.441),
    '41110': (-77.715, 34.142),
    '41109': (-77.300, 34.484),
    '41035': (-77.281, 34.476),
    '41036': (-76.949, 34.207),
    '41159': (-76.944, 34.211),
    '41007': (-76.5, 34.2),
    
    # Central buoys
    'jprn7': (-75.5870, 35.9120),
    '41017': (-75.1000, 35.4000),
    '41015': (-75.3000, 35.4000),
    '41120': (-75.2580, 35.2580),
    'dsln7': (-75.2970, 35.1530),
    '41025': (-75.4540, 35.0100),
    '44095': (-75.3300, 35.7500),
    '44086': (-75.3300, 35.7500),
    
    # Northern buoys
    '44006': (-75.4000, 36.3000),
    '44079': (-75.5930, 36.1750),
    '44056': (-75.7140, 36.2000),
    '44100': (-75.5930, 36.2580),
    '44019': (-75.2000, 36.4000),
}



crop_concatenated_files_by_spatial_mask(
                                        grid=['grid1'],
                                        variable= ['tp','dp','phs0','phs1','phs2','phs3','ptp0','ptp1','ptp2','ptp3','dp0','dp1','dp2','dp3','tm02','dm'],
                                        buoy_coordinates=buoys,
                                        )


In [ ]:
## Plot the final points before merging grids
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.feature import LAND, OCEAN, COASTLINE, STATES, BORDERS
import numpy as np

# Load the final points before merging
p1 = xr.open_dataset('/home/grupos/geocean/montanoj/ShoreShop2026/outputs/cropped_variables/hs_grid1_masked.nc')
p2 = xr.open_dataset('/home/grupos/geocean/montanoj/ShoreShop2026/outputs/cropped_variables/hs_grid2_masked.nc')
p3 = xr.open_dataset('/home/grupos/geocean/montanoj/ShoreShop2026/outputs/cropped_variables/hs_grid3_masked.nc')  
p4 = xr.open_dataset('/home/grupos/geocean/montanoj/ShoreShop2026/outputs/cropped_variables/hs_grid4_masked.nc')

# Define North Carolina extent (approximate bounds)
nc_extent = [-81.0, -74.0, 32, 37.5]  # [lon_min, lon_max, lat_min, lat_max]

# List of datasets and titles
datasets = [p1,p2,p3,p4]

# Also create a combined plot showing all grids together
fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})

# Add map features
ax.add_feature(COASTLINE, linewidth=0.8, edgecolor='black')
ax.add_feature(STATES, linewidth=0.8, edgecolor='gray', linestyle='--')
ax.add_feature(LAND, facecolor='lightgray', alpha=0.5)
ax.add_feature(OCEAN, facecolor='lightblue', alpha=0.3)

# Set extent to North Carolina region
ax.set_extent(nc_extent, crs=ccrs.PlateCarree())

# Plot all grids with different colors
colors = ['red', 'blue', 'green', 'orange']
labels = ['Grid 1', 'Grid 2', 'Grid 3', 'Grid 4']

for ds, color, label in zip(datasets, colors, labels):
    ax.scatter(ds.lon.values, ds.lat.values, 
              c=color, s=8, alpha=0.6, 
              transform=ccrs.PlateCarree(),
              label=label, edgecolors='none')

# Add gridlines
ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')

# Add legend
ax.legend(loc='upper right', fontsize=10)

ax.set_title('All Grids Overlay - North Carolina Region', 
            fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


### 5. Merge the Grids (Overlapping areas)

Here depending of the distance of each point to de border of the grid soe coeficient

#### Test overlapping blending is working properly

#### Final Merge

In [1]:
# Memory management utilities to prevent kernel crashes
import psutil
import os
import gc  # For garbage collection
# Set more aggressive garbage collection
gc.set_threshold(700, 10, 10)

def print_memory_usage(label=""):
    """Print current memory usage"""
    try:
        process = psutil.Process(os.getpid())
        mem_info = process.memory_info()
        mem_gb = mem_info.rss / (1024**3)
        if label:
            print(f"  {label}: {mem_gb:.2f} GB")
        else:
            print(f"  Memory usage: {mem_gb:.2f} GB")
    except:
        pass

def force_gc():
    """Force garbage collection"""
    collected = gc.collect()
    return collected

print("Memory management utilities loaded")
print_memory_usage("Initial")

Memory management utilities loaded
  Initial: 0.07 GB


In [2]:

import numpy as np
import pandas as pd
import xarray as xr
import gc  # For garbage collection
from pathlib import Path
from utils.outputs_grids import merge_multiple_grids

GRIDS = {
    "grid1": "grid1",
    "grid2": "grid2",
    "grid3": "grid3",
    "grid4": "grid4",
}

def validate_netcdf_file(file_path, var_name=None):
    """
    Validate that a NetCDF file can be opened and read.
    Returns (is_valid, error_message)
    """
    try:
        with xr.open_dataset(file_path) as ds:
            # Try to access basic info
            _ = ds.sizes
            # If var_name specified, try to access it
            if var_name and var_name in ds.data_vars:
                # Try to read a small sample (first site, first time)
                sample = ds[var_name].isel(site=0, time=0).values
                _ = sample
        return True, None
    except Exception as e:
        return False, str(e)

def cleanup_corrupted_temp_files(directory, var_name=None):
    """
    Find and delete corrupted temporary NetCDF files.
    Returns count of deleted files.
    """
    deleted_count = 0
    temp_files = list(directory.glob("temp_*.nc"))
    
    if len(temp_files) == 0:
        return 0
    
    print(f"\n{'='*80}")
    print(f"Checking for corrupted temporary files...")
    print(f"{'='*80}")
    print(f"Found {len(temp_files)} temporary file(s) to check")
    
    for temp_file in temp_files:
        is_valid, error = validate_netcdf_file(temp_file, var_name)
        if not is_valid:
            print(f"  ✗ Corrupted: {temp_file.name}")
            print(f"    Error: {error}")
            try:
                temp_file.unlink()
                deleted_count += 1
                print(f"    ✓ Deleted")
            except Exception as e:
                print(f"    ⚠ Could not delete: {e}")
        else:
            print(f"  ✓ Valid: {temp_file.name}")
    
    if deleted_count > 0:
        print(f"\n  Cleaned up {deleted_count} corrupted temporary file(s)")
    else:
        print(f"\n  All temporary files are valid")
    print(f"{'='*80}\n")
    
    return deleted_count

# ============================================================================
# MERGE ALL GRIDS SETTINGS
# ============================================================================
# Convert both to Path objects
CONCATENATED_DIR = Path("/home/grupos/geocean/montanoj/ShoreShop2026/outputs/cropped_variables")
FINAL_MERGED_DIR = Path("/home/grupos/geocean/montanoj/ShoreShop2026/outputs/merged_grids")

# Now these will work
CONCATENATED_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MERGED_DIR.mkdir(parents=True, exist_ok=True)

# And this will work too
if not CONCATENATED_DIR.exists():
    raise ValueError(f"Concatenated grids directory not found: {CONCATENATED_DIR}")

# Blending parameters (same as verification)
BLEND_STEEPNESS = 10.0
TOLERANCE_DEG = 0.001

# Variables to process (discover from concatenated files or specify)
# If None, will discover from concatenated files
VARIABLES_TO_PROCESS = None  # Set to None to auto-discover, or specify like ['hs', 'tp', 'dm', ...]

# Grid names in order (will merge sequentially: grid1+grid2, then result+grid3, then result+grid4)
GRID_NAMES = list(GRIDS.keys())  # ['grid1', 'grid2', 'grid3', 'grid4']
# ============================================================================
print(f"{'='*80}")
print(f"MERGING ALL GRIDS USING CONCATENATED FILES")
print(f"{'='*80}")
print(f"Concatenated files directory: {CONCATENATED_DIR}")
print(f"Output directory: {FINAL_MERGED_DIR}")
print(f"Blend steepness: {BLEND_STEEPNESS}")
print(f"Tolerance: {TOLERANCE_DEG} degrees")
print(f"Grids to merge (in order): {', '.join(GRID_NAMES)}")
print(f"{'='*80}")

# Check if concatenated directory exists
if not CONCATENATED_DIR.exists():
    raise ValueError(f"Concatenated grids directory not found: {CONCATENATED_DIR}")

# Discover variables if not specified
if VARIABLES_TO_PROCESS is None:
    print(f"\nDiscovering variables from concatenated files...")
    variables_found = set()
    for grid_name in GRID_NAMES:
        for file in CONCATENATED_DIR.glob(f"*_{grid_name}_masked.nc"):
            # Extract variable name from filename (e.g., hs_grid1_masked.nc -> hs)
            # Split: remove '_masked' suffix, then split by '_' and take everything before grid name
            stem = file.stem.replace('_masked', '')  # Remove '_masked' suffix
            var_name = stem.rsplit(f'_{grid_name}', 1)[0]  # Split from right, take first part
            variables_found.add(var_name)
    
    # Define priority order for variables
    PRIORITY_VARIABLES = ['hs', 'tp', 'dm','dp']  # Process these first in this order
    # PRIORITY_VARIABLES = ['tp0', 'tp1', 'tp2', 'tp3','dp0', 'dp1', 'dp2', 'dp3']  # Process these first in this order
    
    # Sort variables: priority variables first (in specified order), then rest alphabetically
    priority_vars = []
    other_vars = []
    
    for var in PRIORITY_VARIABLES:
        if var in variables_found:
            priority_vars.append(var)
    
    for var in sorted(variables_found):
        if var not in PRIORITY_VARIABLES:
            other_vars.append(var)
    
    VARIABLES_TO_PROCESS = priority_vars + other_vars
    print(f"  Found variables: {', '.join(VARIABLES_TO_PROCESS)}")
    print(f"  Processing order: Priority variables first ({', '.join(priority_vars)}), then others alphabetically")
else:
    print(f"\nUsing specified variables: {', '.join(VARIABLES_TO_PROCESS)}")

if len(VARIABLES_TO_PROCESS) == 0:
    raise ValueError("No variables found to process!")

# Check which variables have already been processed
print(f"\n{'='*80}")
print(f"Checking for already processed variables...")
print(f"{'='*80}")
already_processed = []
to_process = []

for var_name in VARIABLES_TO_PROCESS:
    final_output_file = FINAL_MERGED_DIR / f"{var_name}_merged_all.nc"
    if final_output_file.exists():
        already_processed.append(var_name)
        print(f"  ✓ {var_name}: Already processed ({final_output_file.name})")
    else:
        to_process.append(var_name)

if len(already_processed) > 0:
    print(f"\n  Skipping {len(already_processed)} already processed variable(s): {', '.join(already_processed)}")

if len(to_process) == 0:
    print(f"\n{'='*80}")
    print(f"All variables have already been processed!")
    print(f"{'='*80}")
    print(f"\nFinal merged files are in: {FINAL_MERGED_DIR}")
    raise SystemExit("No variables to process. All done!")

print(f"\n  Will process {len(to_process)} variable(s): {', '.join(to_process)}")
print(f"{'='*80}")

# Clean up any corrupted temp files before starting
print(f"\n{'='*80}")
print(f"CLEANING UP CORRUPTED TEMPORARY FILES")
print(f"{'='*80}")
cleanup_corrupted_temp_files(FINAL_MERGED_DIR)

# Process each variable
for var_idx, var_name in enumerate(to_process):
    print(f"\n{'='*80}")
    print(f"Processing variable: {var_name} ({var_idx+1}/{len(to_process)})")
    print(f"{'='*80}")
    
    # Output file for final merged result
    final_output_file = FINAL_MERGED_DIR / f"{var_name}_merged_all.nc"
    
    # Double-check if already processed (safety check)
    if final_output_file.exists():
        print(f"  ⏭ Skipping {var_name}: Final merged file already exists")
        print(f"     {final_output_file}")
        continue
    
    # Collect concatenated files for all grids
    grid_files = []
    missing_grids = []
    
    for grid_name in GRID_NAMES:
        concat_file = CONCATENATED_DIR / f"{var_name}_{grid_name}_masked.nc"
        if concat_file.exists():
            grid_files.append(concat_file)
            print(f"  ✓ Found {grid_name}: {concat_file.name}")
        else:
            missing_grids.append(grid_name)
            print(f"  ✗ Missing {grid_name}: {concat_file}")
    
    if len(grid_files) == 0:
        print(f"  ⚠ No files found for {var_name}, skipping...")
        continue
    
    if len(missing_grids) > 0:
        print(f"  ⚠ Warning: {len(missing_grids)} grid(s) missing: {', '.join(missing_grids)}")
        print(f"  Continuing with {len(grid_files)} available grid(s)")
    
    # Initialize variables for cleanup
    current_merged = None
    ds_grid = None
    ds_grid_aligned = None
    
    try:
        # Load first file to detect actual variable name (with proper cleanup)
        with xr.open_dataset(grid_files[0]) as ds_sample:
            data_vars = [v for v in ds_sample.data_vars if v not in ['lon', 'lat', 'coord_x', 'coord_y', 'site_id', 'polygon_lon', 'polygon_lat']]
            actual_var_name = var_name if var_name in data_vars else data_vars[0] if data_vars else var_name
        
        print(f"\n  Actual variable name: {actual_var_name}")
        print(f"  Merging {len(grid_files)} grids sequentially...")
        
        # First, find union of all time ranges and create common time index
        print(f"\n  Finding time ranges across all grids...")
        all_times = []
        time_ranges = []
        for grid_file in grid_files:
            with xr.open_dataset(grid_file) as ds_temp:
                times = pd.to_datetime(ds_temp.time.values)
                time_ranges.append((times[0], times[-1]))
                all_times.extend(times)
                print(f"    {grid_file.name}: {times[0]} to {times[-1]} ({len(times)} timesteps)")
        
        # Create union of all time points (sorted, unique)
        all_times = pd.to_datetime(sorted(set(all_times)))
        common_start = all_times[0]
        common_end = all_times[-1]
        
        print(f"\n  Union time range: {common_start} to {common_end} ({len(all_times)} unique timesteps)")
        print(f"    (This includes all times from all grids, handling gaps)")
        
        # Sequential merging: merge grids one by one
        # Check if we need to resume from a previous step
        # Look for the highest step number with existing temp files (both merged and grid temp files)
        # Also validate that the files are not corrupted
        resume_from_step = None
        for step in range(len(grid_files) - 1, 0, -1):  # Check steps 1 to len-1 (step 0 doesn't create temp files)
            temp_merged_file = FINAL_MERGED_DIR / f"temp_{var_name}_merged_step{step}.nc"
            # Extract grid name from filename (e.g., hs_grid1_masked.nc -> grid1)
            stem_no_masked = grid_files[step].stem.replace('_masked', '')
            grid_name_at_step = stem_no_masked.rsplit('_', 1)[-1]
            temp_grid_file = FINAL_MERGED_DIR / f"temp_{var_name}_{grid_name_at_step}_step{step}.nc"
            if temp_merged_file.exists() and temp_grid_file.exists():
                # Validate both files before using them
                is_valid_merged, error_merged = validate_netcdf_file(temp_merged_file, actual_var_name)
                is_valid_grid, error_grid = validate_netcdf_file(temp_grid_file, actual_var_name)
                
                if is_valid_merged and is_valid_grid:
                    resume_from_step = step
                    print(f"\n  ↻ Detected resume point: Step {step + 1} (found valid temporary files)")
                    print(f"     Will resume from: {temp_merged_file.name} and {temp_grid_file.name}")
                    break
                else:
                    # Files are corrupted, delete them
                    print(f"\n  ⚠ Detected corrupted temporary files at step {step + 1}, deleting...")
                    if not is_valid_merged:
                        print(f"     {temp_merged_file.name}: {error_merged}")
                        try:
                            temp_merged_file.unlink()
                        except:
                            pass
                    if not is_valid_grid:
                        print(f"     {temp_grid_file.name}: {error_grid}")
                        try:
                            temp_grid_file.unlink()
                        except:
                            pass
                    print(f"     Will recreate files from scratch")
        
        for merge_step, grid_file in enumerate(grid_files):
            # Extract grid name from filename (e.g., hs_grid1_masked.nc -> grid1)
            stem_no_masked = grid_file.stem.replace('_masked', '')
            grid_name = stem_no_masked.rsplit('_', 1)[-1]
            
            # If resuming, skip steps before the resume point
            if resume_from_step is not None and merge_step < resume_from_step:
                print(f"\n    Step {merge_step + 1}: Skipping (already completed)")
                continue
            
            # Ensure previous datasets are closed before proceeding
            if ds_grid is not None:
                try:
                    ds_grid.close()
                except:
                    pass
                ds_grid = None
            if ds_grid_aligned is not None:
                try:
                    ds_grid_aligned.close()
                except:
                    pass
                ds_grid_aligned = None
            
            # If this is the resume step, load current_merged from temp file
            temp_next_file = FINAL_MERGED_DIR / f"temp_{var_name}_{grid_name}_step{merge_step}.nc"
            is_resuming = resume_from_step is not None and merge_step == resume_from_step and temp_next_file.exists()
            
            if is_resuming:
                temp_merged_file = FINAL_MERGED_DIR / f"temp_{var_name}_merged_step{resume_from_step}.nc"
                if temp_merged_file.exists():
                    print(f"\n    Step {merge_step + 1}: Resuming from checkpoint...")
                    # Close any existing current_merged before loading new one
                    if current_merged is not None:
                        try:
                            current_merged.close()
                        except:
                            pass
                    current_merged = xr.open_dataset(temp_merged_file)
                    print(f"      ↻ Loaded previous result: {current_merged.sizes['site']} sites, {current_merged.sizes['time']} timesteps")
            else:
                # Load grid and reindex to common time index (handles gaps by filling with NaN)
                print(f"\n    Step {merge_step + 1}: Loading {grid_name}...")
                ds_grid = xr.open_dataset(grid_file)
                
                # Create common time coordinate
                common_time = xr.DataArray(all_times, dims=['time'], name='time')
                # Reindex to common time (missing times will be NaN)
                ds_grid_aligned = ds_grid.reindex(time=common_time, method=None)
                
                # Check for gaps
                times_grid = pd.to_datetime(ds_grid.time.values)
                missing_times = set(all_times) - set(times_grid)
                if len(missing_times) > 0:
                    print(f"      ⚠ Note: {len(missing_times)} timesteps missing in {grid_name} (will be NaN in merged result)")
            
            if merge_step == 0:
                # First grid: use aligned dataset
                print(f"      Aligning to common time index...")
                current_merged = ds_grid_aligned
                print(f"      ✓ Loaded {grid_name}: {current_merged.sizes['site']} sites, {current_merged.sizes['time']} timesteps")
                
                # Filter out unwanted variables from first grid
                vars_to_drop = []
                for var in current_merged.data_vars:
                    if var in ['coord_x', 'coord_y', 'site_id'] or var.startswith('weight_'):
                        vars_to_drop.append(var)
                
                if vars_to_drop:
                    current_merged = current_merged.drop_vars(vars_to_drop)
                    print(f"      Removed extra variables: {', '.join(vars_to_drop)}")
                
                # Clear attributes
                current_merged.attrs = {}
                    
                # Force garbage collection to free memory after merge
                collected = gc.collect()
                if collected > 0:
                    print(f"      Freed {collected} objects from memory")
                # Close original dataset (aligned version is now current_merged)
                ds_grid.close()
                ds_grid = None
                ds_grid_aligned = None  # Don't close, it's now current_merged
            else:
                # Merge current merged result with next grid
                print(f"      Preparing merge with {grid_name}...")
                
                # Check for existing temporary files (resume from checkpoint)
                temp_current_file = FINAL_MERGED_DIR / f"temp_{var_name}_merged_step{merge_step}.nc"
                temp_next_file = FINAL_MERGED_DIR / f"temp_{var_name}_{grid_name}_step{merge_step}.nc"
                
                # Check if we can resume from existing temp files (validate first)
                can_resume = False
                if temp_current_file.exists() and temp_next_file.exists():
                    # Validate both files
                    is_valid_current, error_current = validate_netcdf_file(temp_current_file, actual_var_name)
                    is_valid_next, error_next = validate_netcdf_file(temp_next_file, actual_var_name)
                    
                    if is_valid_current and is_valid_next:
                        can_resume = True
                        print(f"      ↻ Resuming from existing temporary files...")
                        print(f"         Found: {temp_current_file.name}")
                        print(f"         Found: {temp_next_file.name}")
                    else:
                        # Files are corrupted, delete them
                        print(f"      ⚠ Temporary files are corrupted, will recreate...")
                        if not is_valid_current:
                            print(f"         {temp_current_file.name}: {error_current}")
                            try:
                                temp_current_file.unlink()
                            except:
                                pass
                        if not is_valid_next:
                            print(f"         {temp_next_file.name}: {error_next}")
                            try:
                                temp_next_file.unlink()
                            except:
                                pass
                
                if can_resume:
                    # Close current_merged if it's open (it might be from previous step)
                    if current_merged is not None:
                        try:
                            current_merged.close()
                        except:
                            pass
                        current_merged = None
                else:
                    # Create temporary file for current merged result
                    if temp_current_file.exists():
                        print(f"      ↻ Found existing temp file: {temp_current_file.name}")
                        if current_merged is not None:
                            try:
                                current_merged.close()
                            except:
                                pass
                            current_merged = None
                    else:
                        print(f"      Saving current merged result to temp file...")
                        current_merged.to_netcdf(temp_current_file)
                        current_merged.close()
                        current_merged = None
                        gc.collect()  # Force garbage collection after saving
                    
                    # Create temporary file for aligned next grid
                    if temp_next_file.exists():
                        print(f"      ↻ Found existing temp file: {temp_next_file.name}")
                    else:
                        print(f"      Saving aligned grid to temp file...")
                        ds_grid_aligned.to_netcdf(temp_next_file)
                        gc.collect()  # Force garbage collection after saving
                    
                    # Close grid datasets now that we've saved them
                    if ds_grid is not None:
                        ds_grid.close()
                        ds_grid = None
                    if ds_grid_aligned is not None:
                        ds_grid_aligned.close()
                        ds_grid_aligned = None
                
                # Merge current merged result with next grid
                print(f"      Merging grids...")
                try:
                    current_merged = merge_multiple_grids(
                        grid_files=[temp_current_file, temp_next_file],
                        var_name=actual_var_name,
                        steepness=BLEND_STEEPNESS,
                        tolerance_deg=TOLERANCE_DEG,
                        output_file=None,  # Don't save intermediate files
                        use_quality_checks=False  # Disabled to reduce memory usage
                    )
                    print(f"      ✓ Merged result: {current_merged.sizes['site']} sites, {current_merged.sizes['time']} timesteps")
                    
                    # Filter out unwanted variables - keep only original input variables
                    # Remove: coord_x, coord_y, weight_* variables, site_id
                    vars_to_drop = []
                    for var in current_merged.data_vars:
                        if var in ['coord_x', 'coord_y', 'site_id'] or var.startswith('weight_'):
                            vars_to_drop.append(var)
                    
                    if vars_to_drop:
                        current_merged = current_merged.drop_vars(vars_to_drop)
                        print(f"      Removed extra variables: {', '.join(vars_to_drop)}")
                    
                    # Clear attributes that were added by merge_multiple_grids
                    current_merged.attrs = {}
                    
                    # Force garbage collection to free memory after merge
                    collected = gc.collect()
                    if collected > 0:
                        print(f"      Freed {collected} objects from memory")
                except (RuntimeError, OSError) as e:
                    error_msg = str(e)
                    if "HDF error" in error_msg or "NetCDF" in error_msg:
                        print(f"      ✗ HDF/NetCDF error during merge - temp files may be corrupted")
                        print(f"         Error: {error_msg}")
                        # Delete corrupted temp files and try to recreate from source
                        print(f"      Attempting to recreate temp files from source...")
                        try:
                            if temp_current_file.exists():
                                temp_current_file.unlink()
                            if temp_next_file.exists():
                                temp_next_file.unlink()
                        except:
                            pass
                        # Recreate temp files - need to reload from original sources
                        # This is a fallback - ideally we'd have the data in memory, but we'll reload
                        print(f"      Reloading data to recreate temp files...")
                        # Reload current merged from previous step or source
                        if merge_step > 0:
                            # Need to go back and recreate from step 0
                            raise RuntimeError(f"Cannot recover from corrupted temp files at step {merge_step + 1}. Please delete all temp files and restart from beginning.")
                    else:
                        # Re-raise if it's a different error
                        raise
                
                # Clean up temporary files immediately after merge
                if temp_current_file.exists():
                    try:
                        temp_current_file.unlink()
                    except:
                        pass
                if temp_next_file.exists():
                    try:
                        temp_next_file.unlink()
                    except:
                        pass
                
                # Force garbage collection after merge
                gc.collect()
        
        # Save final merged result
        print(f"\n  Saving final merged file...")
        if current_merged is not None:
            # Prepare encoding for compression (matching individual grid files)
            encoding = {}
            valid_encoding_keys = {
                "zlib", "complevel", "shuffle", "fletcher32", "contiguous",
                "chunksizes", "dtype", "_FillValue"
            }
            
            # Set encoding for all data variables (ensure float32 and compression)
            for var_name_enc in current_merged.data_vars:
                var = current_merged[var_name_enc]
                
                # Set encoding with compression
                var_enc = var.encoding.copy() if hasattr(var, 'encoding') and var.encoding else {}
                
                # Only set float32 encoding for numeric variables (not strings)
                if np.issubdtype(var.dtype, np.floating):
                    # Ensure float32 dtype
                    if var.dtype != np.float32:
                        current_merged[var_name_enc] = var.astype(np.float32)
                    # Set encoding with compression for numeric variables
                    var_enc["dtype"] = "float32"
                    var_enc["zlib"] = True
                    var_enc["complevel"] = 4  # Compression level (1-9, 4 is a good balance)
                    var_enc["shuffle"] = True  # Enable shuffle filter for better compression
                # For non-numeric variables (strings, etc.), don't set dtype or compression
                # They will use their default encoding
                
                # Clean encoding to only include valid keys
                cleaned_enc = {k: v for k, v in var_enc.items() if k in valid_encoding_keys}
                encoding[var_name_enc] = cleaned_enc
            
            # Preserve coordinate encodings
            for coord_name in current_merged.coords:
                coord = current_merged[coord_name]
                coord_enc = coord.encoding.copy() if hasattr(coord, 'encoding') and coord.encoding else {}
                cleaned_coord_enc = {k: v for k, v in coord_enc.items() if k in valid_encoding_keys}
                encoding.setdefault(coord_name, cleaned_coord_enc)
            
            # Save with compression encoding using atomic write pattern
            # Write to temporary file first, then rename atomically to avoid permission issues
            import time
            temp_output_file = final_output_file.parent / f".{final_output_file.name}.tmp"
            max_retries = 3
            retry_delay = 1.0  # seconds
            
            try:
                # Clean up any existing temporary files first
                if temp_output_file.exists():
                    try:
                        temp_output_file.unlink()
                    except:
                        pass
                
                # Clean up any existing final file that might be locked
                if final_output_file.exists():
                    try:
                        # Try to close any open handles by forcing garbage collection
                        gc.collect()
                        time.sleep(0.5)  # Brief delay to allow file handles to close
                        final_output_file.unlink()
                        print(f"    ↻ Removed existing file: {final_output_file.name}")
                    except Exception as cleanup_error:
                        print(f"    ⚠ Could not remove existing file (may be locked): {cleanup_error}")
                        # Continue anyway - the temp file write should work
                
                # Write to temporary file
                for attempt in range(max_retries):
                    try:
                        current_merged.to_netcdf(temp_output_file, encoding=encoding)
                        break  # Success, exit retry loop
                    except (PermissionError, OSError) as e:
                        if attempt < max_retries - 1:
                            print(f"    ↻ Retry {attempt + 1}/{max_retries} after permission error...")
                            time.sleep(retry_delay * (attempt + 1))
                            gc.collect()  # Force cleanup before retry
                        else:
                            raise  # Re-raise on final attempt
                
                # Verify the temporary file was created successfully and has reasonable size
                if not temp_output_file.exists():
                    raise RuntimeError("Temporary file was not created after save")
                
                file_size_mb = temp_output_file.stat().st_size / (1024 * 1024)
                if file_size_mb < 0.1:  # Less than 100KB is suspicious for merged data
                    temp_output_file.unlink()  # Clean up small file
                    raise RuntimeError(f"File size is suspiciously small: {file_size_mb:.1f} MB")
                
                # Atomically rename temporary file to final file
                # This is an atomic operation on most filesystems
                temp_output_file.replace(final_output_file)
                
                # Verify final file exists
                if not final_output_file.exists():
                    raise RuntimeError("Final file was not created after atomic rename")
                
                # Print summary
                print(f"\n  ✓ Successfully merged {len(grid_files)} grids for {var_name}")
                print(f"    Final output: {final_output_file}")
                print(f"    File size: {file_size_mb:.1f} MB (compressed, float32)")
                print(f"    Sites: {current_merged.sizes['site']}")
                print(f"    Timesteps: {current_merged.sizes['time']}")
                if 'time' in current_merged.coords:
                    times = pd.to_datetime(current_merged.time.values)
                    print(f"    Time range: {times[0]} to {times[-1]}")
                
            except Exception as save_error:
                # Clean up temporary file if it exists
                if temp_output_file.exists():
                    try:
                        temp_output_file.unlink()
                        print(f"    ↻ Cleaned up temporary file: {temp_output_file.name}")
                    except:
                        pass
                
                # Try to clean up final file if it exists and is corrupted
                if final_output_file.exists():
                    try:
                        # Check if file is suspiciously small (likely corrupted)
                        file_size_mb = final_output_file.stat().st_size / (1024 * 1024)
                        if file_size_mb < 0.1:
                            final_output_file.unlink()
                            print(f"    ⚠ Deleted corrupted file: {final_output_file.name}")
                    except:
                        pass
                
                # Re-raise the error so it gets caught by outer exception handler
                raise save_error
            finally:
                if current_merged is not None:
                    current_merged.close()
                    current_merged = None
        
        # Final cleanup
        gc.collect()
        
    except Exception as e:
        print(f"  ✗ Error processing {var_name}: {e}")
        import traceback
        traceback.print_exc()
        
        # Ensure cleanup on error
        if current_merged is not None:
            try:
                current_merged.close()
            except:
                pass
        if ds_grid is not None:
            try:
                ds_grid.close()
            except:
                pass
        if ds_grid_aligned is not None:
            try:
                ds_grid_aligned.close()
            except:
                pass
        
        gc.collect()
        continue

print(f"\n{'='*80}")
print(f"MERGING COMPLETE")
print(f"{'='*80}")
print(f"\nFinal merged files saved to: {FINAL_MERGED_DIR}")
print(f"One file per variable containing all grids merged sequentially.")


MERGING ALL GRIDS USING CONCATENATED FILES
Concatenated files directory: /home/grupos/geocean/montanoj/ShoreShop2026/outputs/cropped_variables
Output directory: /home/grupos/geocean/montanoj/ShoreShop2026/outputs/merged_grids
Blend steepness: 10.0
Tolerance: 0.001 degrees
Grids to merge (in order): grid1, grid2, grid3, grid4

Discovering variables from concatenated files...
  Found variables: hs, tp, dm, dp, dp0, dp1, dp2, dp3, phs0, phs1, phs2, phs3, ptp0, ptp1, ptp2, ptp3, spr0, spr1, spr2, spr3, tm02
  Processing order: Priority variables first (hs, tp, dm, dp), then others alphabetically

Checking for already processed variables...
  ✓ hs: Already processed (hs_merged_all.nc)
  ✓ tp: Already processed (tp_merged_all.nc)
  ✓ dm: Already processed (dm_merged_all.nc)
  ✓ dp: Already processed (dp_merged_all.nc)
  ✓ dp0: Already processed (dp0_merged_all.nc)
  ✓ dp1: Already processed (dp1_merged_all.nc)
  ✓ dp2: Already processed (dp2_merged_all.nc)
  ✓ dp3: Already processed (dp3_mer

/vols/abedul/home/grupos/geocean/montanoj/ShoreShop2026/utils/outputs_grids.py:1022: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={'time': 10000})
/vols/abedul/home/grupos/geocean/montanoj/ShoreShop2026/utils/outputs_grids.py:1070: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Sites: {ds.dims['site']}, Timesteps: {ds.dims['time']}")
/vols/abedul/home/grupos/geocean/montanoj/ShoreShop2026/utils/outputs_grids.py:1022: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  

  Found 2397 unique point locations

Processing points and blending overlapping regions for variable: spr...


  spr: merging points: 100%|██████████| 2397/2397 [1:23:20<00:00,  2.09s/it]



  Merged 2397 points
  Time range: 1980-01-01 00:00:00 to 2023-12-31 23:00:00
  Total timesteps: 385714

      ✓ Merged result: 2397 sites, 385714 timesteps
      Removed extra variables: coord_x, coord_y, site_id, weight_grid1, weight_grid2
      Freed 690 objects from memory

    Step 3: Loading grid4...
      ⚠ Note: 754 timesteps missing in grid4 (will be NaN in merged result)
      Preparing merge with grid4...
      ↻ Resuming from existing temporary files...
         Found: temp_spr3_merged_step2.nc
         Found: temp_spr3_grid4_step2.nc
      Merging grids...

Merging 2 grids for variable: spr
Quality checks: DISABLED

[1/2] Loading temp_spr3_merged_step2.nc...
  ✓ Created convex hull polygon for grid1 (18 vertices)
  Sites: 2397, Timesteps: 385714

[2/2] Loading temp_spr3_grid4_step2.nc...
  ✓ Created convex hull polygon for grid2 (11 vertices)
  Sites: 843, Timesteps: 385714



/vols/abedul/home/grupos/geocean/montanoj/ShoreShop2026/utils/outputs_grids.py:1070: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Sites: {ds.dims['site']}, Timesteps: {ds.dims['time']}")
/vols/abedul/home/grupos/geocean/montanoj/ShoreShop2026/utils/outputs_grids.py:1022: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(grid_file, chunks={'time': 10000})
/vols/abedul/home/grupos/geocean/montanoj/ShoreShop2026/utils/outputs_grids.py:1070: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping fr

  Found 2594 unique point locations

Processing points and blending overlapping regions for variable: spr...


  spr: merging points: 100%|██████████| 2594/2594 [53:39<00:00,  1.24s/it]  



  Merged 2594 points
  Time range: 1980-01-01 00:00:00 to 2023-12-31 23:00:00
  Total timesteps: 385714

      ✓ Merged result: 2594 sites, 385714 timesteps
      Removed extra variables: coord_x, coord_y, site_id, weight_grid1, weight_grid2
      Freed 809 objects from memory

  Saving final merged file...
    ↻ Removed existing file: spr3_merged_all.nc

  ✓ Successfully merged 3 grids for spr3
    Final output: /home/grupos/geocean/montanoj/ShoreShop2026/outputs/merged_grids/spr3_merged_all.nc
    File size: 2944.0 MB (compressed, float32)
    Sites: 2594
    Timesteps: 385714
    Time range: 1980-01-01 00:00:00 to 2023-12-31 23:00:00

MERGING COMPLETE

Final merged files saved to: /home/grupos/geocean/montanoj/ShoreShop2026/outputs/merged_grids
One file per variable containing all grids merged sequentially.


### 6. Wave Buoy Comparison

#### Bulk Parameters

In [ ]:
# Wave buoy coordinates - deduplicated (removed all duplicates)
from pathlib import Path
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
from math import radians, sin, cos, sqrt, atan2
import sys
sys.path.append('/home/grupos/geocean/montanoj/ShoreShop2026')
from utils.plotting import create_text_with_metrics, fast_density_estimation
import numpy as np


buoys = {
    # # Southern buoys
    # 'SSBN7': (-78.484, 33.838),
    # '41119': (-78.483, 33.842),
    # 'OCPN7': (-78.147, 33.911),
    # '41108': (-78.016, 33.721), 
    # '41013': (-77.764, 33.441),
    # '41110': (-77.715, 34.142),
    # '41109': (-77.300, 34.484),
    # '41035': (-77.281, 34.476),
    # '41036': (-76.949, 34.207),
    # '41159': (-76.944, 34.211),
    # '41007': (-76.5, 34.2),
    
    # # Central buoys
    # 'jprn7': (-75.5870, 35.9120),
    # '41017': (-75.1000, 35.4000),
    # '41015': (-75.3000, 35.4000),
    # '41120': (-75.2580, 35.2580),
    # 'dsln7': (-75.2970, 35.1530),
    # '41025': (-75.4540, 35.0100),
    # '44095': (-75.3300, 35.7500),
    # '44086': (-75.3300, 35.7500),
    
    # Northern buoys
    # '44006': (-75.4000, 36.3000),
    # '44079': (-75.5930, 36.1750),
    '44056': (-75.7140, 36.2000),
    # '44100': (-75.5930, 36.2580),
    # '44019': (-75.2000, 36.4000),
}

print(f"Total unique buoys: {len(buoys)}")
print(f"Buoy IDs: {sorted(buoys.keys())}")

# Compare time series and scatter plots: Merged data vs Buoy data
import matplotlib.pyplot as plt
import os
from math import radians, sin, cos, sqrt, atan2
import sys
sys.path.append('/home/grupos/geocean/montanoj/ShoreShop2026')
from utils.plotting import create_text_with_metrics, fast_density_estimation

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate great circle distance between two points in km"""
    R = 6371  # Earth radius in km
    lat1_rad = radians(lat1)
    lat2_rad = radians(lat2)
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(lat1_rad) * cos(lat2_rad) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    distance = R * c
    return distance

# Load merged datasets
print("Loading merged datasets...")
merged_hs_file = Path('/home/grupos/geocean/montanoj/ShoreShop2026/outputs/merged_grids/hs_merged_all.nc')
merged_hs_file = Path('/home/grupos/geocean/montanoj/ShoreShop2026/outputs/merged_grids/hs_merged_all.nc')
# merged_tp_file = Path('/home/grupos/geocean/montanoj/ShoreShop2026/outputs/merged_grids/merged_all_grids/tp_merged_all_float32.nc')
# merged_dm_file = Path('/home/grupos/geocean/montanoj/ShoreShop2026/outputs/merged_grids/merged_all_grids/dm_merged_all_float32.nc')

if not merged_hs_file.exists():
    print(f"Error: Merged file not found: {merged_hs_file}")
    print("Please run the merge cell (Cell 9) first.")
else:
    ds_merged_hs = xr.open_dataset(merged_hs_file)
    # ds_merged_tp = xr.open_dataset(merged_tp_file)
    # ds_merged_dm = xr.open_dataset(merged_dm_file)
    
    # Get merged coordinates
    if 'coord_x' in ds_merged_hs.coords:
        merged_lon = ds_merged_hs.coord_x.values
        merged_lat = ds_merged_hs.coord_y.values
    else:
        merged_lon = ds_merged_hs.lon.values
        merged_lat = ds_merged_hs.lat.values
    
    print(f"Loaded merged datasets: {len(merged_lon)} sites")
    
    # Process each buoy
    for buoy_id, (buoy_lon, buoy_lat) in buoys.items():
        print(f"\n{'='*60}")
        print(f"Processing buoy: {buoy_id}")
        print(f"Buoy coordinates: lon={buoy_lon}°, lat={buoy_lat}°")
        print(f"{'='*60}")
        
        try:
            # Load buoy data
            buoy_file = f'inputs/buoy_data/buoy_{buoy_id}_bulk_parameters.pkl'
            if not os.path.exists(buoy_file):
                print(f"  ⚠ Buoy file not found: {buoy_file}")
                continue
            
            buoy_data = pd.read_pickle(buoy_file)
       
            
            # Find closest site in merged data
            distances_km = np.array([haversine_distance(buoy_lat, buoy_lon, lat, lon)
                                      for lat, lon in zip(merged_lat, merged_lon)])
            
            closest_site_idx = np.argmin(distances_km)
            closest_distance = distances_km[closest_site_idx]
            closest_lon = merged_lon[closest_site_idx]
            closest_lat = merged_lat[closest_site_idx]
            
            print(f"  Closest site: index={closest_site_idx}, distance={closest_distance:.3f} km")
            print(f"  Site coordinates: lon={closest_lon:.3f}°, lat={closest_lat:.3f}°")
            
            # Extract time series from merged data
            hs_merged = ds_merged_hs.hs.isel(site=closest_site_idx).values
            # tp_merged = ds_merged_tp.tp.isel(site=closest_site_idx).values
            # dm_merged = ds_merged_dm.dm.isel(site=closest_site_idx).values
            
            # Get time axis from merged data
            merged_times = pd.to_datetime(ds_merged_hs.time.values)
            
            # Align time indices between buoy and merged data
            buoy_times = pd.to_datetime(buoy_data.index)
            
            # Round buoy times to nearest hour for matching
            buoy_times_rounded = buoy_times.round('1H')
            
            # Find common times
            common_times = merged_times[merged_times.isin(buoy_times_rounded)]
            
            if len(common_times) == 0:
                print(f"  ⚠ No common times found between buoy and merged data")
                continue
            
            print(f"  Found {len(common_times)} common time steps")
            
            # Get indices for common times
            merged_time_indices = [np.where(merged_times == t)[0][0] for t in common_times if t in merged_times]
            buoy_time_indices = [np.where(buoy_times_rounded == t)[0][0] for t in common_times if t in buoy_times_rounded]
            
            # Extract aligned data
            hs_merged_aligned = hs_merged[merged_time_indices]
            # tp_merged_aligned = tp_merged[merged_time_indices]
            # dm_merged_aligned = dm_merged[merged_time_indices]
            
            # Extract buoy data
            hs_buoy = buoy_data['Hs_Buoy'].values[buoy_time_indices] if 'Hs_Buoy' in buoy_data.columns else None
            # tp_buoy = buoy_data['Tp_Buoy'].values[buoy_time_indices] if 'Tp_Buoy' in buoy_data.columns else None
            # dp_buoy = buoy_data['Dir_Buoy'].values[buoy_time_indices] if 'Dir_Buoy' in buoy_data.columns else None
            
            # Create time series plot (3 panels: hs, tp, dir)
            print("  Creating time series plot...")
            fig1, axes1 = plt.subplots(3, 1, figsize=(14, 10))
            fig1.patch.set_facecolor('black')
            
            # Panel 1: Hs time series
            ax1 = axes1[0]
            ax1.set_facecolor('black')
            if hs_buoy is not None:
                ax1.plot(common_times, hs_buoy, color='white', label='Buoy', linewidth=1.5, alpha=0.9)
            ax1.plot(common_times, hs_merged_aligned, color='#FF69B4', label='Merged', linewidth=1.5, alpha=0.8)
            ax1.set_ylabel('Hs [m]', color='white', fontsize=12)
            ax1.set_xlabel('Time', color='white', fontsize=12)
            ax1.tick_params(colors='white')
            ax1.grid(True, alpha=0.3, color='white')
            ax1.legend(loc='upper left', facecolor='black', edgecolor='white', labelcolor='white')
            ax1.spines['bottom'].set_color('white')
            ax1.spines['top'].set_color('white')
            ax1.spines['right'].set_color('white')
            ax1.spines['left'].set_color('white')
            
            # # Panel 2: Tp time series
            # ax2 = axes1[1]
            # ax2.set_facecolor('black')
            # if tp_buoy is not None:
            #     ax2.plot(common_times, tp_buoy, color='white', label='Buoy', linewidth=1.5, alpha=0.9)
            # ax2.plot(common_times, tp_merged_aligned, color='#FF69B4', label='Merged', linewidth=1.5, alpha=0.8)
            # ax2.set_ylabel('Tp [s]', color='white', fontsize=12)
            # ax2.set_xlabel('Time', color='white', fontsize=12)
            # ax2.tick_params(colors='white')
            # ax2.grid(True, alpha=0.3, color='white')
            # ax2.legend(loc='upper left', facecolor='black', edgecolor='white', labelcolor='white')
            # ax2.spines['bottom'].set_color('white')
            # ax2.spines['top'].set_color('white')
            # ax2.spines['right'].set_color('white')
            # ax2.spines['left'].set_color('white')
            
            # # Panel 3: Dir time series
            # ax3 = axes1[2]
            # ax3.set_facecolor('black')
            # if dp_buoy is not None:
            #     ax3.plot(common_times, dp_buoy, color='white', label='Buoy', linewidth=1.5, alpha=0.9)
            # ax3.plot(common_times, dm_merged_aligned, color='#FF69B4', label='Merged', linewidth=1.5, alpha=0.8)
            # ax3.set_ylabel('Dir [°]', color='white', fontsize=12)
            # ax3.set_xlabel('Time', color='white', fontsize=12)
            # ax3.tick_params(colors='white')
            # ax3.grid(True, alpha=0.3, color='white')
            # ax3.legend(loc='upper left', facecolor='black', edgecolor='white', labelcolor='white')
            # ax3.spines['bottom'].set_color('white')
            # ax3.spines['top'].set_color('white')
            # ax3.spines['right'].set_color('white')
            # ax3.spines['left'].set_color('white')
            
            fig1.suptitle(f'Wave Validation {buoy_id} - Merged Data', color='white', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
            
            # Create scatter plot (3 panels: hs, tp, dir)
            print("  Creating scatter plot...")
            fig2, axes2 = plt.subplots(2, 2, figsize=(14, 12))
            fig2.patch.set_facecolor('black')
            
            # Panel 1: Hs Buoy vs Hs Merged
            ax1 = axes2[0, 0]
            ax1.set_facecolor('black')
            if hs_buoy is not None:
                # Use density coloring
                finite_mask = np.isfinite(hs_buoy) & np.isfinite(hs_merged_aligned)
                if finite_mask.sum() > 0:
                    density = fast_density_estimation(hs_buoy[finite_mask], hs_merged_aligned[finite_mask])
                    scatter = ax1.scatter(hs_buoy[finite_mask], hs_merged_aligned[finite_mask], 
                                         c=density, cmap='plasma', s=1, alpha=0.6)
                    cbar = plt.colorbar(scatter, ax=ax1)
                    cbar.set_label('Density', color='white')
                    cbar.ax.tick_params(colors='white')
                    
                    # Add metrics text
                    metrics_text = create_text_with_metrics(hs_buoy[finite_mask], hs_merged_aligned[finite_mask])
                    ax1.text(0.05, 0.95, metrics_text, transform=ax1.transAxes,
                            verticalalignment='top', color='white', fontsize=9,
                            bbox=dict(boxstyle='round', facecolor='black', edgecolor='white', alpha=0.8))
            
            max_hs = max(np.nanmax(hs_buoy) if hs_buoy is not None else 0, 
                         np.nanmax(hs_merged_aligned) if hs_merged_aligned is not None else 0)
            ax1.plot([0, max_hs], [0, max_hs], 'w--', linewidth=1.5, alpha=0.8)
            ax1.set_xlabel('Hs - Buoy [m]', color='white', fontsize=11)
            ax1.set_ylabel('Hs - Merged [m]', color='white', fontsize=11)
            ax1.tick_params(colors='white')
            ax1.grid(True, alpha=0.3, color='white')
            ax1.spines['bottom'].set_color('white')
            ax1.spines['top'].set_color('white')
            ax1.spines['right'].set_color('white')
            ax1.spines['left'].set_color('white')
            
            # # Panel 2: Tp Buoy vs Tp Merged
            # ax2 = axes2[0, 1]
            # ax2.set_facecolor('black')
            # if tp_buoy is not None:
            #     finite_mask = np.isfinite(tp_buoy) & np.isfinite(tp_merged_aligned)
            #     if finite_mask.sum() > 0:
            #         density = fast_density_estimation(tp_buoy[finite_mask], tp_merged_aligned[finite_mask])
            #         scatter = ax2.scatter(tp_buoy[finite_mask], tp_merged_aligned[finite_mask],
            #                              c=density, cmap='plasma', s=1, alpha=0.6)
            #         cbar = plt.colorbar(scatter, ax=ax2)
            #         cbar.set_label('Density', color='white')
            #         cbar.ax.tick_params(colors='white')
                    
            #         metrics_text = create_text_with_metrics(tp_buoy[finite_mask], tp_merged_aligned[finite_mask])
            #         ax2.text(0.05, 0.95, metrics_text, transform=ax2.transAxes,
            #                 verticalalignment='top', color='white', fontsize=9,
            #                 bbox=dict(boxstyle='round', facecolor='black', edgecolor='white', alpha=0.8))
            
            # max_tp = max(np.nanmax(tp_buoy) if tp_buoy is not None else 0,
            #              np.nanmax(tp_merged_aligned) if tp_merged_aligned is not None else 0)
            # ax2.plot([0, max_tp], [0, max_tp], 'w--', linewidth=1.5, alpha=0.8)
            # ax2.set_xlabel('Tp - Buoy [s]', color='white', fontsize=11)
            # ax2.set_ylabel('Tp - Merged [s]', color='white', fontsize=11)
            # ax2.tick_params(colors='white')
            # ax2.grid(True, alpha=0.3, color='white')
            # ax2.spines['bottom'].set_color('white')
            # ax2.spines['top'].set_color('white')
            # ax2.spines['right'].set_color('white')
            # ax2.spines['left'].set_color('white')
            
            # # Panel 3: Dir Buoy vs Dir Merged
            # ax3 = axes2[1, 0]
            # ax3.set_facecolor('black')
            # if dp_buoy is not None:
            #     finite_mask = np.isfinite(dp_buoy) & np.isfinite(dm_merged_aligned)
            #     if finite_mask.sum() > 0:
            #         density = fast_density_estimation(dp_buoy[finite_mask], dm_merged_aligned[finite_mask])
            #         scatter = ax3.scatter(dp_buoy[finite_mask], dm_merged_aligned[finite_mask],
            #                              c=density, cmap='plasma', s=1, alpha=0.6)
            #         cbar = plt.colorbar(scatter, ax=ax3)
            #         cbar.set_label('Density', color='white')
            #         cbar.ax.tick_params(colors='white')
                    
            #         metrics_text = create_text_with_metrics(dp_buoy[finite_mask], dm_merged_aligned[finite_mask])
            #         ax3.text(0.05, 0.95, metrics_text, transform=ax3.transAxes,
            #                 verticalalignment='top', color='white', fontsize=9,
            #                 bbox=dict(boxstyle='round', facecolor='black', edgecolor='white', alpha=0.8))
            
            # max_dir = max(np.nanmax(dp_buoy) if dp_buoy is not None else 0,
            #               np.nanmax(dm_merged_aligned) if dm_merged_aligned is not None else 0)
            # ax3.plot([0, max_dir], [0, max_dir], 'w--', linewidth=1.5, alpha=0.8)
            # ax3.set_xlabel('Dir - Buoy [°]', color='white', fontsize=11)
            # ax3.set_ylabel('Dir - Merged [°]', color='white', fontsize=11)
            # ax3.tick_params(colors='white')
            # ax3.grid(True, alpha=0.3, color='white')
            # ax3.spines['bottom'].set_color('white')
            # ax3.spines['top'].set_color('white')
            # ax3.spines['right'].set_color('white')
            # ax3.spines['left'].set_color('white')
            
            # # Panel 4: Empty or statistics summary
            # ax4 = axes2[1, 1]
            # ax4.set_facecolor('black')
            # ax4.axis('off')
            
            # Print summary statistics
            stats_text = f"Validation Statistics\n{'='*40}\n"
            if hs_buoy is not None:
                finite_mask = np.isfinite(hs_buoy) & np.isfinite(hs_merged_aligned)
                if finite_mask.sum() > 0:
                    stats_text += f"\nHs:\n"
                    stats_text += f"  N: {finite_mask.sum()}\n"
                    stats_text += f"  RMSE: {np.sqrt(np.nanmean((hs_buoy[finite_mask] - hs_merged_aligned[finite_mask])**2)):.3f} m\n"
                    stats_text += f"  Bias: {np.nanmean(hs_merged_aligned[finite_mask] - hs_buoy[finite_mask]):.3f} m\n"
                    stats_text += f"  R²: {np.corrcoef(hs_buoy[finite_mask], hs_merged_aligned[finite_mask])[0,1]**2:.3f}\n"
            
            if tp_buoy is not None:
                finite_mask = np.isfinite(tp_buoy) & np.isfinite(tp_merged_aligned)
                if finite_mask.sum() > 0:
                    stats_text += f"\nTp:\n"
                    stats_text += f"  N: {finite_mask.sum()}\n"
                    stats_text += f"  RMSE: {np.sqrt(np.nanmean((tp_buoy[finite_mask] - tp_merged_aligned[finite_mask])**2)):.3f} s\n"
                    stats_text += f"  Bias: {np.nanmean(tp_merged_aligned[finite_mask] - tp_buoy[finite_mask]):.3f} s\n"
                    stats_text += f"  R²: {np.corrcoef(tp_buoy[finite_mask], tp_merged_aligned[finite_mask])[0,1]**2:.3f}\n"
            
            if dp_buoy is not None:
                finite_mask = np.isfinite(dp_buoy) & np.isfinite(dm_merged_aligned)
                if finite_mask.sum() > 0:
                    stats_text += f"\nDir:\n"
                    stats_text += f"  N: {finite_mask.sum()}\n"
                    # Handle circular statistics for direction
                    diff = dm_merged_aligned[finite_mask] - dp_buoy[finite_mask]
                    diff = np.where(diff > 180, diff - 360, diff)
                    diff = np.where(diff < -180, diff + 360, diff)
                    stats_text += f"  RMSE: {np.sqrt(np.nanmean(diff**2)):.1f}°\n"
                    stats_text += f"  Bias: {np.nanmean(diff):.1f}°\n"
                    stats_text += f"  R²: {np.corrcoef(dp_buoy[finite_mask], dm_merged_aligned[finite_mask])[0,1]**2:.3f}\n"
            
            ax4.text(0.1, 0.5, stats_text, transform=ax4.transAxes,
                    verticalalignment='center', color='white', fontsize=10,
                    family='monospace',
                    bbox=dict(boxstyle='round', facecolor='black', edgecolor='white', alpha=0.8))
            
            fig2.suptitle(f'Wave Validation Scatter {buoy_id} - Merged Data', color='white', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
            
            print(f"  ✓ Plots created for buoy {buoy_id}")
            
        except Exception as e:
            print(f"  ✗ Error processing buoy {buoy_id}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Close datasets
    ds_merged_hs.close()
    # ds_merged_tp.close()
    # ds_merged_dm.close()

print("\n" + "="*60)
print("All buoy comparisons completed!")
print("="*60)

#### Reconstruction Partitions

### 7. Generate GeoJson Statistics